<div style="text-align: center;">
  <span style="
      color: lightskyblue;
      font-family: 'Segoe UI', 'Noto Sans KR', 'Malgun Gothic', sans-serif;
      font-weight: 700;
      font-size: 48px;
      letter-spacing: 0.5px;
      line-height: 1.2;
  ">
    Demo File 
  </span>
</div>

Raw version of demoFile.py

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Import Library
</span>

In [ ]:
import rby1_sdk as rby
import numpy as np
import logging
import time
import pyrealsense2 as rs
import cv2

import sys
from pathlib import Path
# Path.cwd()           : folder "260723", 
# Path.cwd().parents[0]: folder "Demonstration", 
# Path.cwd().parents[1]: folder "RBY1_library_ysw"
sys.path.append(str(Path.cwd().parents[1]))                  
import rby1_lib_ysw as ysw

np.set_printoptions(suppress=True, precision=4, linewidth=300)  

# In .ipynb, show logging 
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s",
    # format="%(asctime)s [%(levelname)s] %(message)s",
    stream=sys.stdout,
    force=True,
)

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Initialization [Connect + PowerON]
</span>

In [2]:
# ==========================================================================================================
# Robot Initialization
# ==========================================================================================================
robotConfig = ysw.setRobotConfig(ysw.MODE_DIRECT_UPC, ysw.MODEL_A)          # For Real Robot + UPC

robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_AND_POWER)
if (robot != None):
    model = robot.model()
    
time.sleep(1)


# ==========================================================================================================
# Gripper Initialization
# Gripper Setting (Need to hard check of gripper ping test)
# ==========================================================================================================
gripper = ysw.Gripper()
gripper.connect_hardCheck(
    device_name=ysw.GRIPPER_DEVICE_NAME, 
    protocol_version=ysw.DXL_PROTOCOL_VERSION, 
    baudrate=ysw.DXL_BAUDRATE,
    retry_interval=1.0,
    max_retry=5
)


# ==========================================================================================================
# Depth Camera Initialization
# D435i:  Head
# D405_R: Right Wrist
# D405_L: Left Wrist
# ==========================================================================================================
d435i = ysw.RealSenseDepthCamera(rgb_on=True, depth_on=True, pixel=(640,480), fps=30, serial_number=ysw.HEAD_D435I_SERIAL)
d435i.initialize()
d405_R = ysw.RealSenseDepthCamera(rgb_on=True, depth_on=True, pixel=(640,480), fps=30, serial_number=ysw.RIGHT_D405_SERIAL)
d405_R.initialize()
d405_L = ysw.RealSenseDepthCamera(rgb_on=True, depth_on=True, pixel=(640,480), fps=30, serial_number=ysw.LEFT_D405_SERIAL)
d405_L.initialize()

*************************** [Initialization Starts] ****************************

[INFO] SUCCESS: Connected to the Robot!


[INFO] SUCCESS: Power ON the Robot!
[INFO] SUCCESS: Servo ON the Robot!
[INFO] SUCCESS: Enable the Control Manager!
[INFO] SUCCESS: Enable the Grippers!

*************************** [Intialization is Done] ****************************



*********************** [Gripper Hard Connection Starts] ***********************

[Gripper] Trying to connect gripper... attempt=1
************************* [Gripper Connection Starts] **************************

[INFO] SUCCESS: open the port: /dev/rby1_gripper
[INFO] SUCCESS: change the baudrate 2000000

************************* [Gripper Connection is Done] *************************



************************** [Gripper Ping Test Starts] **************************

[INFO] [DXL ID:0] SUCCESS: Dynamixel model number is 1030
[INFO] [DXL ID:1] SUCCESS: Dynamixel model number is 1030

************************* [Gripper Ping Test is Done] **************************



[Gripper] connect=True, ping_test=True

********************** [Gripper Hard Connection 

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Tuning Parameters
</span>

In [4]:
position_control_minimum_time = 8.5      # [sec]

# Move to Table
vision_record_duration_tableDistance = 5.0    # [sec]
offset_table_distance = 0.37                  # [m]
vision_record_duration_checkBox = 5.0         # [sec]
vision_record_duration_checkObject = 5.0      # [sec]
box_minimum_area = 8000                       # [pixel^2]
box_maximum_distance = 1.2                    # [m]
object_minimum_area = 50                      # [pixel^2]
object_maximum_area = 6000                    # [pixel^2]
object_maximum_distance = 1.2                 # [m]

# minimum_time_scaler = 0.5                # For short movement
grasp_current_obj = 0.1      # [A]
object_floating_z = 0.125     # [m]
offset_gripper_open = 0.015  # [m]
manual_offset_downToGrasp = -0.008       # [m] for z-direction offset when grasp


<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Posture [Initial]
</span>

In [4]:
desired_pos = ysw.PACK_POS  
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=position_control_minimum_time, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************

[ERROR] Failed to conduct [move_position_22dof].finish_code = FinishCode.ControlManagerFault


False

<span style="
    color:gray;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Table Distance Detection [Visualization]
</span>

In [ ]:
# ============================================================
# ROI setting 
# Pixel Origin: (0,0) = 왼쪽 위 꼭짓점
# Posture: PACK_POS
# ============================================================
width = 640
height = 480
pixel_roi_x = [140, width - 140]      
pixel_roi_y = [320, height]    
# pixel_roi_x = None            # None이면 x 전체 사용
# pixel_roi_y = None            # None이면 y 전체 사용
    
    
# ============================================================
# Get the Distance of Center Pixel 
# Table distance is approximated as the depth of center pixel
# ============================================================

d435i.startPipeline()

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)
        
        h, w, _ = color_image.shape
        
    
        # ROI center pixel in full image coordinate
        if pixel_roi_x is None:
            x_min = 0
            x_max = w
        else:
            x_min, x_max = pixel_roi_x
            x_min = int(np.clip(x_min, 0, w - 1))
            x_max = int(np.clip(x_max, 0, w))

        if pixel_roi_y is None:
            y_min = 0
            y_max = h
        else:
            y_min, y_max = pixel_roi_y
            y_min = int(np.clip(y_min, 0, h - 1))
            y_max = int(np.clip(y_max, 0, h))
            
        x_roi_center = (x_min + x_max) // 2
        y_roi_center = (y_min + y_max) // 2

        distance = d435i.getDepth(depth_frame, (x_roi_center, y_roi_center))   # unit = [m]
        
        
        # Draw ROI box on full image
        cv2.rectangle(
            color_image,
            (x_min, y_min),
            (x_max, y_max),
            (255, 0, 0),
            2
        )
        cv2.putText(
            color_image,                   
            f"ROI Box",
            (20, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 0, 0),
            2
        )
        
        # Draw ROI center point on full image
        cv2.circle(color_image, (x_roi_center, y_roi_center), 5, (0, 0, 255), -1)
        cv2.putText(
            color_image,                   
            f"[ROI center point] Depth: {distance:.3f} m",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 0, 255),
            2
        )

        d435i.imshow("color_image", color_image)
        d435i.imshow("depth_image", depth_colorMap)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Table Distance Detection [5 seconds]
</span>

In [4]:
# ============================================================
# ROI setting 
# Pixel Origin: (0,0) = 왼쪽 위 꼭짓점
# Posture: PACK_POS
# ============================================================
width = 640
height = 480
pixel_roi_x = [140, width - 140]      
pixel_roi_y = [320, height]    
# pixel_roi_x = None            # None이면 x 전체 사용
# pixel_roi_y = None            # None이면 y 전체 사용
    
    
# ============================================================
# Get the Distance of Center Pixel 
# Table distance is approximated as the depth of center pixel
# ============================================================

record_duration = vision_record_duration_tableDistance      # [sec]
distance_list = []

d435i.startPipeline()

try:
    start_time = time.time()
    
    while True:
        elapsed_time = time.time() - start_time
        
        if elapsed_time >= record_duration:
            break
        
        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        h, w, _ = color_image.shape
        
        # ROI center pixel in full image coordinate
        if pixel_roi_x is None:
            x_min = 0
            x_max = w
        else:
            x_min, x_max = pixel_roi_x
            x_min = int(np.clip(x_min, 0, w - 1))
            x_max = int(np.clip(x_max, 0, w))

        if pixel_roi_y is None:
            y_min = 0
            y_max = h
        else:
            y_min, y_max = pixel_roi_y
            y_min = int(np.clip(y_min, 0, h - 1))
            y_max = int(np.clip(y_max, 0, h))
            
        x_roi_center = (x_min + x_max) // 2
        y_roi_center = (y_min + y_max) // 2

        distance = d435i.getDepth(depth_frame, (x_roi_center, y_roi_center))   # unit = [m]
        distance_list.append(distance)
        print(f"Table Distance: {distance:.3f} [m]",)

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()
    

# Average Distance
distance_avg = np.mean(distance_list)
print(f"\nTable Average Distance: {distance_avg} [m]")

moved_distance_to_table = distance_avg - offset_table_distance
print(f"moved_distance_to_table: {moved_distance_to_table} [m]")

[INFO] [Head D435i] Pipeline started.
Table Distance: 0.647 [m]
Table Distance: 0.647 [m]
Table Distance: 0.647 [m]
Table Distance: 0.647 [m]
Table Distance: 0.638 [m]
Table Distance: 0.638 [m]
Table Distance: 0.639 [m]
Table Distance: 0.639 [m]
Table Distance: 0.638 [m]
Table Distance: 0.639 [m]
Table Distance: 0.638 [m]
Table Distance: 0.638 [m]
Table Distance: 0.638 [m]
Table Distance: 0.638 [m]
Table Distance: 0.639 [m]
Table Distance: 0.639 [m]
Table Distance: 0.638 [m]
Table Distance: 0.639 [m]
Table Distance: 0.637 [m]
Table Distance: 0.638 [m]
Table Distance: 0.640 [m]
Table Distance: 0.638 [m]
Table Distance: 0.640 [m]
Table Distance: 0.638 [m]
Table Distance: 0.638 [m]
Table Distance: 0.639 [m]
Table Distance: 0.638 [m]
Table Distance: 0.640 [m]
Table Distance: 0.638 [m]
Table Distance: 0.638 [m]
Table Distance: 0.638 [m]
Table Distance: 0.639 [m]
Table Distance: 0.638 [m]
Table Distance: 0.638 [m]
Table Distance: 0.638 [m]
Table Distance: 0.638 [m]
Table Distance: 0.637 [m]


<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Posture [DEMO POSE]
</span>

In [5]:
desired_pos = ysw.DEMO_BASIC_POS_260826  
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=position_control_minimum_time, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Move to Table
</span>

In [9]:
ysw.mobile_linear_distance(
    robot,
    distance_m=moved_distance_to_table,
    max_speed=0.25,
    accel=0.8,
    tolerance_m=0.01
)

distance=0.000[m], speed=0.016[m/s], pos_error=1.261[m], yaw_error=0.00[deg]
distance=0.010[m], speed=0.144[m/s], pos_error=1.251[m], yaw_error=-0.02[deg]
distance=0.051[m], speed=0.250[m/s], pos_error=1.209[m], yaw_error=-0.01[deg]
distance=0.103[m], speed=0.250[m/s], pos_error=1.157[m], yaw_error=-0.11[deg]
distance=0.156[m], speed=0.250[m/s], pos_error=1.105[m], yaw_error=0.08[deg]
distance=0.208[m], speed=0.250[m/s], pos_error=1.053[m], yaw_error=0.09[deg]
distance=0.260[m], speed=0.250[m/s], pos_error=1.001[m], yaw_error=-0.00[deg]
distance=0.312[m], speed=0.250[m/s], pos_error=0.949[m], yaw_error=-0.01[deg]
distance=0.365[m], speed=0.250[m/s], pos_error=0.896[m], yaw_error=-0.03[deg]
distance=0.417[m], speed=0.250[m/s], pos_error=0.844[m], yaw_error=-0.01[deg]
distance=0.469[m], speed=0.250[m/s], pos_error=0.792[m], yaw_error=-0.01[deg]
distance=0.521[m], speed=0.250[m/s], pos_error=0.739[m], yaw_error=-0.02[deg]
distance=0.573[m], speed=0.250[m/s], pos_error=0.688[m], yaw_error=

True

<span style="
    color:gray;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Check Black & Red Boxes and Objects [Visualization]
</span>

In [ ]:
# # ============================================================
# # ROI setting 
# # Pixel Origin: (0,0) = 왼쪽 위 꼭짓점
# # Posture: DEMO_BASIC_POSE_260826
# # ============================================================
# width = 640
# height = 480
# pixel_roi_x = None            # None이면 x 전체 사용
# pixel_roi_y = [150, height]  


# # ============================================================
# # Get the Information about Black & Red Collecting Boxes
# # ============================================================

# rv = d435i.startPipeline()

# # if rv is False:
# #     print(f"[WARNING] Camera start failed. Try recovery...")
# #     rv = d435i.recoverPipeline(
# #         align_enable=True,
# #         try_hardware_reset=True,
# #         wait_after_reset_sec=5.0,
# #     )
# # if rv is False:
# #     raise RuntimeError(f"Camera Recovery failed.")   

# try:
#     while True:
#         color_frame, depth_frame = d435i.getRawFrames()

#         if color_frame is None or depth_frame is None:
#             continue

#         color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
#         depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)

#         # ----------------------------------------------------
#         # Keep raw image for detection
#         # Do not detect on already-annotated image
#         # ----------------------------------------------------
#         color_image_raw = color_image.copy()
        
#         detected_blackBoxes, color_image_forBlackBox, mask_blackBox = d435i.detectColorObjectContour3D(
#             color_image=color_image_raw.copy(),
#             depth_frame=depth_frame,
#             mask_list=ysw.HSV_MASK_BLACK_BOX,
#             minimum_area=box_minimum_area,       
#             maximum_depth=box_maximum_distance,                   # ignore distant object larger than x[m] 
#             pixel_roi_x=pixel_roi_x,
#             pixel_roi_y=pixel_roi_y,
#         )
#         detected_redBoxes, color_image_forRedBox, mask_redBox = d435i.detectColorObjectContour3D(
#             color_image=color_image_raw.copy(),
#             depth_frame=depth_frame,
#             mask_list=ysw.HSV_MASK_RED_BOX,
#             minimum_area=box_minimum_area,       
#             maximum_depth=box_maximum_distance,                   # ignore distant object larger than x[m] 
#             pixel_roi_x=pixel_roi_x,
#             pixel_roi_y=pixel_roi_y,
#         )
#         detected_blackObjects, color_image_forBlackObj, mask_blackObj = d435i.detectColorObjectContour3D(
#             color_image=color_image_raw.copy(),
#             depth_frame=depth_frame,
#             mask_list=ysw.HSV_MASK_BLACK_BOX,
#             minimum_area=object_minimum_area, 
#             maximum_area=object_maximum_area,      
#             maximum_depth=box_maximum_distance,                   # ignore distant object larger than x[m] 
#             pixel_roi_x=pixel_roi_x,
#             pixel_roi_y=pixel_roi_y,
#         )
#         detected_redObjects, color_image_forRedObj, mask_redObj = d435i.detectColorObjectContour3D(
#             color_image=color_image_raw.copy(),
#             depth_frame=depth_frame,
#             mask_list=ysw.HSV_MASK_RED,
#             minimum_area=object_minimum_area, 
#             maximum_area=object_maximum_area,         
#             maximum_depth=box_maximum_distance,                   # ignore distant object larger than x[m] 
#             pixel_roi_x=pixel_roi_x,
#             pixel_roi_y=pixel_roi_y,
#         )
        
#         for obj in detected_blackBoxes:
#             print(
#                 "[Black Box]: "
#                 f"[object {obj['id']}] "
#                 f"center_3d={np.round(obj['center_3d'], 3)}, "
#                 f"bbox_3d_size={np.round(obj['bbox_3d_size'], 3)}"
#                 f"area_2d={obj['area_2d']:.1f}"
#             )
#         for obj in detected_redBoxes:
#             print(
#                 "[Red Box]: "
#                 f"[object {obj['id']}] "
#                 f"center_3d={np.round(obj['center_3d'], 3)}, "
#                 f"bbox_3d_size={np.round(obj['bbox_3d_size'], 3)}, "
#                 f"area_2d={obj['area_2d']:.1f}"
#             )
#         for obj in detected_blackObjects:
#             print(
#                 "[Black Objects]: "
#                 f"[object {obj['id']}] "
#                 f"center_3d={np.round(obj['center_3d'], 3)}, "
#                 f"bbox_3d_size={np.round(obj['bbox_3d_size'], 3)}"
#                 f"area_2d={obj['area_2d']:.1f}"
#             )
#         for obj in detected_redObjects:
#             print(
#                 "[Orange Objects]: "
#                 f"[object {obj['id']}] "
#                 f"center_3d={np.round(obj['center_3d'], 3)}, "
#                 f"bbox_3d_size={np.round(obj['bbox_3d_size'], 3)}"
#                 f"area_2d={obj['area_2d']:.1f}"
#             )
        
#         d435i.imshow("blackBox_detection", color_image_forBlackBox)
#         d435i.imshow("redBox_detection", color_image_forRedBox)
#         d435i.imshow("blackObj_detection", color_image_forBlackObj)
#         d435i.imshow("orangeObj_detection", color_image_forRedObj)


#         key = d435i.waitKey(1)
#         if key == 27:
#             break

# finally:
#     d435i.stopPipeline()
#     d435i.destroyAllWindows()

[INFO] [Head D435i] Pipeline started.


[Black Box]: [object 0] center_3d=[-0.18   0.105  0.664], bbox_3d_size=[0.179 0.192 0.157]area_2d=24601.5
[Black Objects]: [object 0] center_3d=[0.266 0.141 0.643], bbox_3d_size=[0.105 0.035 0.042]area_2d=2268.5
[Black Objects]: [object 1] center_3d=[0.152 0.121 0.684], bbox_3d_size=[0.015 0.011 0.007]area_2d=111.0
[Black Objects]: [object 2] center_3d=[0.002 0.118 0.655], bbox_3d_size=[0.038 0.043 0.09 ]area_2d=1198.5
[Black Objects]: [object 3] center_3d=[0.155 0.061 0.74 ], bbox_3d_size=[0.013 0.012 0.006]area_2d=85.5
[Black Objects]: [object 4] center_3d=[-0.069  0.046  0.728], bbox_3d_size=[0.043 0.05  0.02 ]area_2d=1121.0
[Black Objects]: [object 5] center_3d=[0.014 0.044 0.739], bbox_3d_size=[0.041 0.049 0.024]area_2d=1055.0
[Black Box]: [object 0] center_3d=[-0.192  0.101  0.67 ], bbox_3d_size=[0.129 0.19  0.245]area_2d=22051.0
[Red Box]: [object 0] center_3d=[0.29  0.059 0.71 ], bbox_3d_size=[0.125 0.145 0.149], area_2d=10621.5
[Black Objects]: [object 0] center_3d=[0.261 0.14

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Check [Black & Red] Boxes [5 seconds]
</span>

In [6]:
# ============================================================
# ROI setting 
# Pixel Origin: (0,0) = 왼쪽 위 꼭짓점
# Posture: DEMO_BASIC_POSE_260826
# ============================================================
width = 640
height = 480
pixel_roi_x = None            # None이면 x 전체 사용
pixel_roi_y = [150, height]  



# ============================================================
# Get the Information about Black & Red Boxes and Objects
# ============================================================
record_duration = vision_record_duration_checkObject  # [sec]

blackBox_samples = []
redBox_samples = []

d435i.startPipeline()

try:
    start_time = time.time()

    while True:
        elapsed_time = time.time() - start_time

        if elapsed_time >= record_duration:
            break

        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)

        # Detection용 원본 이미지 보존
        color_image_raw = color_image.copy()

        detected_blackBoxes, color_image_forBlackBox, mask_blackBox = d435i.detectColorObjectContour3D(
            color_image=color_image_raw.copy(),
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_BLACK_BOX,
            minimum_area=box_minimum_area,       
            maximum_depth=box_maximum_distance,                   # ignore distant object larger than x[m] 
            pixel_roi_x=pixel_roi_x,
            pixel_roi_y=pixel_roi_y,
        )
        detected_redBoxes, color_image_forRedBox, mask_redBox = d435i.detectColorObjectContour3D(
            color_image=color_image_raw.copy(),
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_RED_BOX,
            minimum_area=box_minimum_area,       
            maximum_depth=box_maximum_distance,                   # ignore distant object larger than x[m] 
            pixel_roi_x=pixel_roi_x,
            pixel_roi_y=pixel_roi_y,
        )

        if len(detected_blackBoxes) > 0:
            largest_blackBox = max(
                detected_blackBoxes,
                key=lambda obj: obj["area_2d"]
            )
            # [Largest area] sample
            blackBox_samples.append({
                "area_2d":      float(largest_blackBox["area_2d"]),
                "pixel_center": np.array(largest_blackBox["pixel_center"], dtype=np.float64),
                "center_3d":    np.array(largest_blackBox["center_3d"], dtype=np.float64),
                "bbox_3d_size": np.array(largest_blackBox["bbox_3d_size"], dtype=np.float64),
                "bbox_3d_min":  np.array(largest_blackBox["bbox_3d_min"], dtype=np.float64),
                "bbox_3d_max":  np.array(largest_blackBox["bbox_3d_max"], dtype=np.float64),
            })
            
        if len(detected_redBoxes) > 0:
            largest_redBox = max(
                detected_redBoxes,
                key=lambda obj: obj["area_2d"]
            )
            # [Largest area] sample
            redBox_samples.append({
                "area_2d":      float(largest_redBox["area_2d"]),
                "pixel_center": np.array(largest_redBox["pixel_center"], dtype=np.float64),
                "center_3d":    np.array(largest_redBox["center_3d"], dtype=np.float64),
                "bbox_3d_size": np.array(largest_redBox["bbox_3d_size"], dtype=np.float64),
                "bbox_3d_min":  np.array(largest_redBox["bbox_3d_min"], dtype=np.float64),
                "bbox_3d_max":  np.array(largest_redBox["bbox_3d_max"], dtype=np.float64),
            })

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()



# =========================
# Average Box
# =========================

blackBox_sample_avg = None
redBox_sample_avg = None

if len(blackBox_samples) > 0:
    blackBox_sample_avg = ysw.average_detectColorObjectContour3D(blackBox_samples)
if len(redBox_samples) > 0:
    redBox_sample_avg = ysw.average_detectColorObjectContour3D(redBox_samples)

print("\n========== Black & Red box Detection Result ==========")

if blackBox_sample_avg is None:
    print("No Black Box detected.")
else:
    print("\n[Black Box]")
    print(f"num_samples             : {blackBox_sample_avg['num_samples']}")
    print(f"area_2d_avg             : {blackBox_sample_avg['area_2d_avg']:.1f}")
    print(f"area_2d_median          : {blackBox_sample_avg['area_2d_median']:.1f}")
    print(f"center_3d_avg           : {np.round(blackBox_sample_avg['center_3d_avg'], 4)} m")
    print(f"center_3d_median        : {np.round(blackBox_sample_avg['center_3d_median'], 4)} m")
    print(f"bbox_3d_size_avg        : {np.round(blackBox_sample_avg['bbox_3d_size_avg'], 4)} m")
    print(f"bbox_3d_size_median     : {np.round(blackBox_sample_avg['bbox_3d_size_median'], 4)} m")
    
if redBox_sample_avg is None:
    print("No Red Box detected.")
else:
    print("\n[Red Box]")
    print(f"num_samples             : {redBox_sample_avg['num_samples']}")
    print(f"area_2d_avg             : {redBox_sample_avg['area_2d_avg']:.1f}")
    print(f"area_2d_median          : {redBox_sample_avg['area_2d_median']:.1f}")
    print(f"center_3d_avg           : {np.round(redBox_sample_avg['center_3d_avg'], 4)} m")
    print(f"center_3d_median        : {np.round(redBox_sample_avg['center_3d_median'], 4)} m")
    print(f"bbox_3d_size_avg        : {np.round(redBox_sample_avg['bbox_3d_size_avg'], 4)} m")
    print(f"bbox_3d_size_median     : {np.round(redBox_sample_avg['bbox_3d_size_median'], 4)} m")  
    
    
# =========================
# Set grasp arm 
# ========================= 
grasp_arm_blackObj = None
grasp_arm_redObj = None

if (blackBox_sample_avg['center_3d_avg'][0] > 0):
    grasp_arm_blackObj = "right"
else:
    grasp_arm_blackObj = "left"
    
if (redBox_sample_avg['center_3d_avg'][0] > 0):
    grasp_arm_redObj = "right"
else:
    grasp_arm_redObj = "left"  

print(f"\ngrasp_arm_blackObj: {grasp_arm_blackObj}")
print(f"grasp_arm_redObj: {grasp_arm_redObj}")

[INFO] [Head D435i] Pipeline started.
[INFO] [Head D435i] Pipeline stopped.

========== Black & Red box Detection Result ==========

[Black Box]
num_samples             : 60
area_2d_avg             : 13455.1
area_2d_median          : 13388.0
center_3d_avg           : [-0.3094  0.035   0.73  ] m
center_3d_median        : [-0.3091  0.035   0.73  ] m
bbox_3d_size_avg        : [0.119  0.1758 0.1498] m
bbox_3d_size_median     : [0.1146 0.1741 0.139 ] m

[Red Box]
num_samples             : 56
area_2d_avg             : 12393.5
area_2d_median          : 12373.8
center_3d_avg           : [0.3506 0.0018 0.7529] m
center_3d_median        : [0.3498 0.0022 0.752 ] m
bbox_3d_size_avg        : [0.1339 0.1725 0.1801] m
bbox_3d_size_median     : [0.1239 0.1733 0.171 ] m

grasp_arm_blackObj: left
grasp_arm_redObj: right


<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Check [Black Leftmost & Red Rightmost] Object [5 seconds]
</span>

In [ ]:
# ============================================================
# ROI setting 
# Pixel Origin: (0,0) = 왼쪽 위 꼭짓점
# Posture: DEMO_BASIC_POSE_260826
# ============================================================
width = 640
height = 480
pixel_roi_x = None            # None이면 x 전체 사용
pixel_roi_y = [150, height]  



# ============================================================
# Get the Information about Black & Red Boxes and Objects
# ============================================================
record_duration = vision_record_duration_checkBox  # [sec]

leftmost_blackObj_samples = []
rightmost_redObj_samples = []

d435i.startPipeline()

try:
    start_time = time.time()

    while True:
        elapsed_time = time.time() - start_time

        if elapsed_time >= record_duration:
            break

        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)

        # Detection용 원본 이미지 보존
        color_image_raw = color_image.copy()

        detected_blackObjects, color_image_forBlackObj, mask_blackObj = d435i.detectColorObjectContour3D(
            color_image=color_image_raw.copy(),
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_BLACK_BOX,
            minimum_area=object_minimum_area, 
            maximum_area=object_maximum_area,      
            maximum_depth=box_maximum_distance,                   # ignore distant object larger than x[m] 
            pixel_roi_x=pixel_roi_x,
            pixel_roi_y=pixel_roi_y,
        )
        detected_redObjects, color_image_forRedObj, mask_redObj = d435i.detectColorObjectContour3D(
            color_image=color_image_raw.copy(),
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_RED,
            minimum_area=object_minimum_area, 
            maximum_area=object_maximum_area,         
            maximum_depth=box_maximum_distance,                   # ignore distant object larger than x[m] 
            pixel_roi_x=pixel_roi_x,
            pixel_roi_y=pixel_roi_y,
        )

        if len(detected_blackObjects) > 0:
            # 카메라 기준 X 좌표가 작을수록 이미지 왼쪽
            # RealSense/D435i 기준: X+ = image right
            blackObj_list_fromLeft = sorted(
                detected_blackObjects,
                key=lambda obj: obj["center_3d"][0],
                reverse=False
            )

            # 제일 왼쪽 물체 하나만 선택
            obj = blackObj_list_fromLeft[0]

            # 제일 왼쪽 물체 sample만 저장
            leftmost_blackObj_samples.append({
                "area_2d":      float(obj["area_2d"]),
                "pixel_center": np.array(obj["pixel_center"], dtype=np.float64),
                "center_3d":    np.array(obj["center_3d"], dtype=np.float64),
                "bbox_3d_size": np.array(obj["bbox_3d_size"], dtype=np.float64),
                "bbox_3d_min":  np.array(obj["bbox_3d_min"], dtype=np.float64),
                "bbox_3d_max":  np.array(obj["bbox_3d_max"], dtype=np.float64),
            })
        
        if len(detected_redObjects) > 0:
            # 카메라 기준 X 좌표가 작을수록 이미지 왼쪽
            # RealSense/D435i 기준: X+ = image right
            redObj_list_fromLeft = sorted(
                detected_redObjects,
                key=lambda obj: obj["center_3d"][0],
                reverse=True
            )

            # 제일 오른쪽 물체 하나만 선택
            obj = redObj_list_fromLeft[0]

            # 제일 오른쪽 물체 sample만 저장
            rightmost_redObj_samples.append({
                "area_2d":      float(obj["area_2d"]),
                "pixel_center": np.array(obj["pixel_center"], dtype=np.float64),
                "center_3d":    np.array(obj["center_3d"], dtype=np.float64),
                "bbox_3d_size": np.array(obj["bbox_3d_size"], dtype=np.float64),
                "bbox_3d_min":  np.array(obj["bbox_3d_min"], dtype=np.float64),
                "bbox_3d_max":  np.array(obj["bbox_3d_max"], dtype=np.float64),
            })

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()

    
    
# =========================
# Average leftmost objects
# =========================
leftmost_blackObj_sample_avg = None
rightmost_redObj_sample_avg = None

if len(leftmost_blackObj_samples) > 0:
    leftmost_blackObj_sample_avg = ysw.average_detectColorObjectContour3D(leftmost_blackObj_samples)
if len(rightmost_redObj_samples) > 0:
    rightmost_redObj_sample_avg = ysw.average_detectColorObjectContour3D(rightmost_redObj_samples)

print("\n========== Detection Result ==========")
print("Only the Leftmost object is used\n")

if leftmost_blackObj_sample_avg is None:
    print("No Black Object detected.")
else:
    print("\n[Leftmost Black Object]")
    print(f"num_samples             : {leftmost_blackObj_sample_avg['num_samples']}")
    print(f"area_2d_avg             : {leftmost_blackObj_sample_avg['area_2d_avg']:.1f}")
    print(f"area_2d_median          : {leftmost_blackObj_sample_avg['area_2d_median']:.1f}")
    print(f"center_3d_avg           : {np.round(leftmost_blackObj_sample_avg['center_3d_avg'], 4)} m")
    print(f"center_3d_median        : {np.round(leftmost_blackObj_sample_avg['center_3d_median'], 4)} m")
    print(f"bbox_3d_size_avg        : {np.round(leftmost_blackObj_sample_avg['bbox_3d_size_avg'], 4)} m")
    print(f"bbox_3d_size_median     : {np.round(leftmost_blackObj_sample_avg['bbox_3d_size_median'], 4)} m")
    
if rightmost_redObj_sample_avg is None:
    print("No Red Object detected.")
else:
    print("\n[Rightmost Red Object]")
    print(f"num_samples             : {rightmost_redObj_sample_avg['num_samples']}")
    print(f"area_2d_avg             : {rightmost_redObj_sample_avg['area_2d_avg']:.1f}")
    print(f"area_2d_median          : {rightmost_redObj_sample_avg['area_2d_median']:.1f}")
    print(f"center_3d_avg           : {np.round(rightmost_redObj_sample_avg['center_3d_avg'], 4)} m")
    print(f"center_3d_median        : {np.round(rightmost_redObj_sample_avg['center_3d_median'], 4)} m")
    print(f"bbox_3d_size_avg        : {np.round(rightmost_redObj_sample_avg['bbox_3d_size_avg'], 4)} m")
    print(f"bbox_3d_size_median     : {np.round(rightmost_redObj_sample_avg['bbox_3d_size_median'], 4)} m") 

[INFO] [Head D435i] Pipeline started.
[INFO] [Head D435i] Pipeline stopped.

========== Detection Result ==========
Only the Leftmost object is used


[Leftmost Black Object]
num_samples             : 132
area_2d_avg             : 499.8
area_2d_median          : 502.5
center_3d_avg           : [ 0.0838 -0.0113  0.804 ] m
center_3d_median        : [ 0.0848 -0.0124  0.805 ] m
bbox_3d_size_avg        : [0.0293 0.0349 0.0286] m
bbox_3d_size_median     : [0.0297 0.0353 0.031 ] m

[Rightmost Red Object]
num_samples             : 130
area_2d_avg             : 699.3
area_2d_median          : 643.2
center_3d_avg           : [0.0048 0.0937 0.688 ] m
center_3d_median        : [0.0104 0.0911 0.688 ] m
bbox_3d_size_avg        : [0.0313 0.0316 0.0303] m
bbox_3d_size_median     : [0.0263 0.0324 0.03  ] m


<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
IK for [Black-Leftmost & Red-Rightmost] Objects
</span>

In [ ]:
# {V}: vision frame
# {H}: "link_head_2" frame
# {S}: "base" frame
# {M}: Goal frame (object) 
# {M1}: Goal frame (object) before open the gripper 
# {M2}: Goal frame (object) after open the gripper 
# {N}: Goal frame (collecting box) 
# {C}: ee_finger_l1/r1 frame
# {D}: final fingertip frame -> same orientation of {C}

leftmost_blackObj_pos    = leftmost_blackObj_sample_avg["center_3d_avg"]
leftmost_blackObj_3Dsize = leftmost_blackObj_sample_avg["bbox_3d_size_avg"]

rightmost_redObj_pos    = rightmost_redObj_sample_avg["center_3d_avg"]
rightmost_redObj_3Dsize = rightmost_redObj_sample_avg["bbox_3d_size_avg"]

blackBox_pos    = blackBox_sample_avg["center_3d_avg"]
blackBox_3Dsize = blackBox_sample_avg["bbox_3d_size_avg"]

redBox_pos    = redBox_sample_avg["center_3d_avg"]
redBox_3Dsize = redBox_sample_avg["bbox_3d_size_avg"]


P_HM_blackObj = ysw.convertPos_vision_to_linkhead2(leftmost_blackObj_pos)
P_HM_redObj   = ysw.convertPos_vision_to_linkhead2(rightmost_redObj_pos)
P_HN_blackBox = ysw.convertPos_vision_to_linkhead2(blackBox_pos)
P_HN_redBox = ysw.convertPos_vision_to_linkhead2(redBox_pos)

fk_SH = ysw.get_FK(robot, "base", "link_head_2")
T_SH = fk_SH["T"]
R_SH = fk_SH["rotation"]
P_SH = fk_SH["position"]



P_HM_blackObj_extended = np.append(P_HM_blackObj, 1.0)        # Make 4x1 vector
P_SM_blackObj = T_SH @ P_HM_blackObj_extended
P_SM_blackObj = P_SM_blackObj[:3]

P_HM_redObj_extended   = np.append(P_HM_redObj, 1.0)
P_SM_redObj = T_SH @ P_HM_redObj_extended
P_SM_redObj = P_SM_redObj[:3]

P_HN_blackBox_extended = np.append(P_HN_blackBox, 1.0)  
P_SN_blackBox = T_SH @ P_HN_blackBox_extended
P_SN_blackBox = P_SN_blackBox[:3]
      
P_HN_redBox_extended   = np.append(P_HN_redBox, 1.0)
P_SN_redBox = T_SH @ P_HN_redBox_extended
P_SN_redBox = P_SN_redBox[:3]



if (grasp_arm_blackObj == "right"):
    R1 = ysw.rotMat_z(90.0*ysw.DEG2RAD)      # "base" frame 기준으로 z축 +90도 회전
    R2 = ysw.rotMat_x(-40.0*ysw.DEG2RAD)     # After R1, converted "link_torso_5" frame 기준으로 x축 -40도 회전
    R_SC_desired_blackObj = R1 @ R2                  
elif (grasp_arm_blackObj == "left"):
    R1 = ysw.rotMat_z(-90.0*ysw.DEG2RAD)     # "base" frame 기준으로 z축 -90도 회전
    R2 = ysw.rotMat_x(40.0*ysw.DEG2RAD)      # After R1, converted "base" frame 기준으로 x축 +40도 회전
    R_SC_desired_blackObj = R1 @ R2      
    
if (grasp_arm_redObj == "right"):
    R1 = ysw.rotMat_z(90.0*ysw.DEG2RAD)      # "base" frame 기준으로 z축 +90도 회전
    R2 = ysw.rotMat_x(-40.0*ysw.DEG2RAD)     # After R1, converted "link_torso_5" frame 기준으로 x축 -40도 회전
    R_SC_desired_redObj = R1 @ R2                  
elif (grasp_arm_redObj == "left"):
    R1 = ysw.rotMat_z(-90.0*ysw.DEG2RAD)     # "base" frame 기준으로 z축 -90도 회전
    R2 = ysw.rotMat_x(40.0*ysw.DEG2RAD)      # After R1, converted "base" frame 기준으로 x축 +40도 회전
    R_SC_desired_redObj = R1 @ R2                   

R_SN_blackBox = R_SC_desired_blackObj
T_SN_blackBox = ysw.make_HT(R_SN_blackBox, P_SN_blackBox)   

R_SN_redBox = R_SC_desired_redObj
T_SN_redBox = ysw.make_HT(R_SN_redBox, P_SN_redBox)  



P_SM1_blackObj = P_SM_blackObj + np.array([0.0, 0.0, +0.1])      # Z-offset before open the gripper
R_SM1_blackObj = R_SC_desired_blackObj
T_SM1_blackObj = ysw.make_HT(R_SM1_blackObj, P_SM1_blackObj)    

P_SM2_blackObj = P_SM_blackObj + np.array([0.0, 0.0, manual_offset_downToGrasp])      # Theoretically manual_offset should be zero
R_SM2_blackObj = R_SC_desired_blackObj
T_SM2_blackObj = ysw.make_HT(R_SM2_blackObj, P_SM2_blackObj)    



P_SM1_redObj = P_SM_redObj + np.array([0.0, 0.0, +0.05])      # Z-offset before open the gripper
R_SM1_redObj = R_SC_desired_redObj
T_SM1_redObj = ysw.make_HT(R_SM1_redObj, P_SM1_redObj)    

P_SM2_redObj = P_SM_redObj + np.array([0.0, 0.0, manual_offset_downToGrasp])      # Theoretically manual_offset should be zero
R_SM2_redObj = R_SC_desired_redObj
T_SM2_redObj = ysw.make_HT(R_SM2_redObj, P_SM2_redObj) 



# "ee_finger_l1/r1" to final end-effector -> should be physically measured
pos_from_eeFinger_to_fingertip = np.array([0, 0, ysw.GRIPPER_Z_OFFSET_FROM_LINK_EE_FINGER]) # P_CD
P_CD = pos_from_eeFinger_to_fingertip
T_CD = ysw.make_HT(np.eye(3), P_CD)    

T_SC_target_before_open_gripper_blackObj = T_SM1_blackObj @ np.linalg.inv(T_CD)
T_SC_target_after_open_gripper_blackObj  = T_SM2_blackObj @ np.linalg.inv(T_CD)

T_SC_target_before_open_gripper_redObj = T_SM1_redObj @ np.linalg.inv(T_CD)
T_SC_target_after_open_gripper_redObj  = T_SM2_redObj @ np.linalg.inv(T_CD)

print("======== For Black Object ========")
print(f"Before open gripper: \n{T_SC_target_before_open_gripper_blackObj}\n")
print(f"After open gripper: \n{T_SC_target_after_open_gripper_blackObj}\n")
print(f"Grasping Arm: {grasp_arm_blackObj}\n")

print("======== For Red Object ========")
print(f"Before open gripper: \n{T_SC_target_before_open_gripper_redObj}\n")
print(f"After open gripper: \n{T_SC_target_after_open_gripper_redObj}\n")
print(f"Grasping Arm: {grasp_arm_redObj}\n")


T_SC_target_for_blackBox = T_SN_blackBox @ np.linalg.inv(T_CD)
print("======== For Black Box ========")
print(f"For Black Box: \n{T_SC_target_for_blackBox}\n")

T_SC_target_for_redBox = T_SN_redBox @ np.linalg.inv(T_CD)
print("======== For Red Box ========")
print(f"For Red Box: \n{T_SC_target_for_redBox}\n")

******************* [FK of 'link_head_2' from 'base' Starts] *******************

[Result]

T:
[[ 0.7314 -0.      0.682   0.022 ]
 [ 0.      1.      0.     -0.    ]
 [-0.682   0.      0.7314  1.4244]
 [ 0.      0.      0.      1.    ]]


position [m]: [ 0.022  -0.      1.4244]
rpy [rad]: [0.     0.7505 0.    ]
rpy [deg]: [ 0.0003 42.9998  0.0011]

******************** [FK of 'link_head_2' from 'base' Ends] ********************



======== For Black Object ========
Before open gripper: 
[[ 0.      0.766  -0.6428  0.6266]
 [-1.      0.     -0.     -0.0668]
 [ 0.      0.6428  0.766   1.0819]
 [ 0.      0.      0.      1.    ]]

After open gripper: 
[[ 0.      0.766  -0.6428  0.6266]
 [-1.      0.     -0.     -0.0668]
 [ 0.      0.6428  0.766   0.9739]
 [ 0.      0.      0.      1.    ]]

Grasping Arm: left

======== For Red Object ========
Before open gripper: 
[[ 0.     -0.766  -0.6428  0.4701]
 [ 1.      0.      0.      0.0122]
 [ 0.     -0.6428  0.766   1.0342]
 [ 0.      0.      0.   

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Pick [Black-Leftmost object] and Place to [Black Box]
</span>

In [9]:
# ======================================
# Move to Pre-grasp pose
# ======================================
pos_before_open_gripper_blackObj = T_SC_target_before_open_gripper_blackObj[:3,3]
ori_before_open_gripper_blackObj = ysw.rotMat_to_rpy(T_SC_target_before_open_gripper_blackObj[:3,:3])       

if (grasp_arm_blackObj == "right"):      # Object is in right side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "base",
                                                "target_link": "ee_finger_r1",
                                                "pos": pos_before_open_gripper_blackObj,                  
                                                "rpy": ori_before_open_gripper_blackObj,                    
                                            },                 
                                        ]
    
elif (grasp_arm_blackObj == "left"):       # Object is in left side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "base",
                                                "target_link": "ee_finger_l1",
                                                "pos": pos_before_open_gripper_blackObj,                  
                                                "rpy": ori_before_open_gripper_blackObj,                    
                                            },                 
                                        ]

# Fix some joint_targets
current_joint_position = ysw.getState(robot).position
joint_targets = {
    "torso_0": current_joint_position[ysw.ALL_JOINT_INDEX["torso_0"]],
    "torso_1": current_joint_position[ysw.ALL_JOINT_INDEX["torso_1"]],
    "torso_2": current_joint_position[ysw.ALL_JOINT_INDEX["torso_2"]],
    "torso_3": current_joint_position[ysw.ALL_JOINT_INDEX["torso_3"]],
}

# [rv = "function"] form is Necessary when "get_return=False"
rv = ysw.positionCtrl_cartesian_wholeBody(
    robot, 
    position_targets=cartesian_position_targets_sample, 
    joint_targets=joint_targets,
    minimum_time=position_control_minimum_time,
    # minimum_time=position_control_minimum_time * minimum_time_scaler,
    control_hold_time=0.5,
    get_return=True
)



# ======================================
# Open Gripper
# ======================================
object_meanDim = np.mean(leftmost_blackObj_3Dsize)       # mean value of 3D size of object to grasp
desiredGripperPos = object_meanDim + offset_gripper_open        # [m]

desiredGripperPos_mm = desiredGripperPos * 1000.0
if desiredGripperPos_mm >= ysw.GRIPPER_POS_RANGE_LINEAR:
    desiredGripperPos_mm = ysw.GRIPPER_POS_RANGE_LINEAR - 8.0

if (grasp_arm_blackObj == "right"):     
    gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_R, operation_mode=ysw.DXL_CURRENT_BASED_POSITION_CONTROL_MODE)
    gripper.current_based_position_control_linear(ysw.GRIPPER_ID_R, current_ampere_mag=0.02, linear_pos=desiredGripperPos_mm)
    time.sleep(1)
    gripper.disable(ysw.GRIPPER_ID_R)
elif (grasp_arm_blackObj == "left"):      
    gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_L, operation_mode=ysw.DXL_CURRENT_BASED_POSITION_CONTROL_MODE)
    gripper.current_based_position_control_linear(ysw.GRIPPER_ID_L, current_ampere_mag=0.02, linear_pos=desiredGripperPos_mm)
    time.sleep(1)
    gripper.disable(ysw.GRIPPER_ID_L)
    
    
    
# ======================================
# Move to grasp pose
# ======================================
pos_after_open_gripper_blackObj = T_SC_target_after_open_gripper_blackObj[:3,3]
ori_after_open_gripper_blackObj = ysw.rotMat_to_rpy(T_SC_target_after_open_gripper_blackObj[:3,:3])       

if (grasp_arm_blackObj == "right"):      # Object is in right side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "base",
                                                "target_link": "ee_finger_r1",
                                                "pos": pos_after_open_gripper_blackObj,                  
                                                "rpy": ori_after_open_gripper_blackObj,                    
                                            },                 
                                        ]
    
elif (grasp_arm_blackObj == "left"):       # Object is in left side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "base",
                                                "target_link": "ee_finger_l1",
                                                "pos": pos_after_open_gripper_blackObj,                  
                                                "rpy": ori_after_open_gripper_blackObj,                    
                                            },                 
                                        ]

# Fix some joint_targets
current_joint_position = ysw.getState(robot).position
joint_targets = {
    "torso_0": current_joint_position[ysw.ALL_JOINT_INDEX["torso_0"]],
    "torso_1": current_joint_position[ysw.ALL_JOINT_INDEX["torso_1"]],
    "torso_2": current_joint_position[ysw.ALL_JOINT_INDEX["torso_2"]],
    "torso_3": current_joint_position[ysw.ALL_JOINT_INDEX["torso_3"]],
}

# [rv = "function"] form is Necessary when "get_return=False"
rv = ysw.positionCtrl_cartesian_wholeBody(
    robot, 
    position_targets=cartesian_position_targets_sample, 
    joint_targets=joint_targets,
    minimum_time=position_control_minimum_time,
    # minimum_time=position_control_minimum_time * minimum_time_scaler,
    control_hold_time=0.5,
    get_return=True
)



# ======================================
# Grasp
# ======================================
if (grasp_arm_blackObj == "right"):   
    gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_R, operation_mode=ysw.DXL_CURRENT_CONTROL_MODE)
    gripper.write_current_ampere(ysw.GRIPPER_ID_R, grasp_current_obj)
elif (grasp_arm_blackObj == "left"):      
    gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_L, operation_mode=ysw.DXL_CURRENT_CONTROL_MODE)
    gripper.write_current_ampere(ysw.GRIPPER_ID_L, grasp_current_obj)
    
time.sleep(1)



# ========================================================
# slightly float the object (to avoid box collision)
# ========================================================
if (grasp_arm_blackObj == "right"):      # Object is in right side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "base",
                                                "target_link": "ee_finger_r1",
                                                "pos": pos_after_open_gripper_blackObj + np.array([0.0, 0.0, object_floating_z]),                  
                                                "rpy": ori_after_open_gripper_blackObj,                    
                                            },                 
                                        ]
    
elif (grasp_arm_blackObj == "left"):       # Object is in left side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "base",
                                                "target_link": "ee_finger_l1",
                                                "pos": pos_after_open_gripper_blackObj + np.array([0.0, 0.0, object_floating_z]),                  
                                                "rpy": ori_after_open_gripper_blackObj,                    
                                            },                 
                                        ]

# Fix some joint_targets
current_joint_position = ysw.getState(robot).position
joint_targets = {
    "torso_0": current_joint_position[ysw.ALL_JOINT_INDEX["torso_0"]],
    "torso_1": current_joint_position[ysw.ALL_JOINT_INDEX["torso_1"]],
    "torso_2": current_joint_position[ysw.ALL_JOINT_INDEX["torso_2"]],
    "torso_3": current_joint_position[ysw.ALL_JOINT_INDEX["torso_3"]],
}

# [rv = "function"] form is Necessary when "get_return=False"
rv = ysw.positionCtrl_cartesian_wholeBody(
    robot, 
    position_targets=cartesian_position_targets_sample, 
    joint_targets=joint_targets,
    minimum_time=position_control_minimum_time,
    # minimum_time=position_control_minimum_time * minimum_time_scaler,
    control_hold_time=0.5,
    get_return=True
)



# ========================================================
# Move to Black Box
# ========================================================
current_pos = pos_after_open_gripper_blackObj + np.array([0.0, 0.0, object_floating_z])

pos_blackBox = T_SC_target_for_blackBox[:3,3].copy()
pos_blackBox[2] = current_pos[2]     # floating 이후의 height 유지
ori_blackBox = ysw.rotMat_to_rpy(T_SC_target_for_blackBox[:3,:3])    
    
if (grasp_arm_blackObj == "right"):      # Object is in right side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "base",
                                                "target_link": "ee_finger_r1",
                                                "pos": pos_blackBox,                   
                                                "rpy": ori_blackBox,                    
                                            },                 
                                        ]
    
elif (grasp_arm_blackObj == "left"):       # Object is in left side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "base",
                                                "target_link": "ee_finger_l1",
                                                "pos": pos_blackBox,                  
                                                "rpy": ori_blackBox,                    
                                            },                 
                                        ]

# Fix some joint_targets
current_joint_position = ysw.getState(robot).position
joint_targets = {
    "torso_0": current_joint_position[ysw.ALL_JOINT_INDEX["torso_0"]],
    "torso_1": current_joint_position[ysw.ALL_JOINT_INDEX["torso_1"]],
    "torso_2": current_joint_position[ysw.ALL_JOINT_INDEX["torso_2"]],
    "torso_3": current_joint_position[ysw.ALL_JOINT_INDEX["torso_3"]],
}

# [rv = "function"] form is Necessary when "get_return=False"
rv = ysw.positionCtrl_cartesian_wholeBody(
    robot, 
    position_targets=cartesian_position_targets_sample, 
    joint_targets=joint_targets,
    minimum_time=position_control_minimum_time,
    # minimum_time=position_control_minimum_time * minimum_time_scaler,
    control_hold_time=0.5,
    get_return=True
)



# ========================================================
# Open Gripper
# ========================================================
if (grasp_arm_blackObj == "right"):     
    gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_R, operation_mode=ysw.DXL_CURRENT_BASED_POSITION_CONTROL_MODE)
    gripper.current_based_position_control_linear(ysw.GRIPPER_ID_R, current_ampere_mag=0.02, linear_pos=desiredGripperPos_mm)
    time.sleep(1)
    gripper.disable(ysw.GRIPPER_ID_R)
elif (grasp_arm_blackObj == "left"):      
    gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_L, operation_mode=ysw.DXL_CURRENT_BASED_POSITION_CONTROL_MODE)
    gripper.current_based_position_control_linear(ysw.GRIPPER_ID_L, current_ampere_mag=0.02, linear_pos=desiredGripperPos_mm)
    time.sleep(1)
    gripper.disable(ysw.GRIPPER_ID_L)

***************************** [Robot State Starts] *****************************

RobotState_A(
  timestamp=datetime.datetime(1970, 1, 1, 13, 19, 39, 710040),
  system_stat=SystemStat(
    cpu_usage=0.0422,
    memory_usage=0.0682,
    uptime=4.8e+04,
    program_uptime=4.8e+04
  ),
  battery_state=BatteryState(
    voltage=57,
    current=3.5,
    level_percent=82
  ),
  power_states=[PowerState(
    state=PowerState.State.PowerOn,
    voltage=0
  ), PowerState(
    state=PowerState.State.PowerOn,
    voltage=0
  ), PowerState(
    state=PowerState.State.PowerOn,
    voltage=0
  ), PowerState(
    state=PowerState.State.PowerOn,
    voltage=57
  )],
  emo_states=[EMOState(
    state=EMOState.State.Released
  )],
  joint_states=[JointState(
    time_since_last_update=datetime.timedelta(microseconds=2),
    is_ready=True,
    fet_state=JointState.FETState.On,
    run_state=JointState.RunState.ControlOn,
    init_state=JointState.InitializationState.Initialized,
    motor_type=1,
    mot


**************** [Cartesian Position Control(Whole-Body) Ends] *****************



********************* [Gripper [DXL ID:1] setting Starts] **********************

[INFO] SUCCESS: [DXL ID:1] Torque is Disabled
[INFO] SUCCESS: [DXL ID:1] Set Current-based Position Control Mode
[INFO] SUCCESS: [DXL ID:1] Torque is Enabled

********************* [Gripper [DXL ID:1] setting is Done] *********************



********** [Gripper [DXL ID:1] Current-based Position Control Starts] **********

Desired Linear Position = 45.96210896014mm
[INFO] [DXL ID:1] SUCCESS: operation mode = 5
[INFO] SUCCESS: [DXL ID:1] Set Current [0.02 A]
[INFO] SUCCESS: [DXL ID:1] Set Goal Position [Tick]: signed=544, uint32=544

*********** [Gripper [DXL ID:1] Current-based Position Control Ends] ***********



********************* [Gripper [DXL ID:1] Disable Starts] **********************

[INFO] SUCCESS: [DXL ID:1] Torque is Disabled

********************* [Gripper [DXL ID:1] Disable is Done] *********************
